# ACTIVE-TB-S: Full 156-Task Gold-Free Testbench Synthesis

This phase consumes only sanitized specifications and generated oracle candidates. Five ablation variants, including typed temporal execution plus replay-checked formal disagreement, are hash-frozen before golden evaluation assets become available.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
markers=list(INPUT.rglob('active_tb_s_synthesis_full_v1/SYNTHESIS_BOUNDARY.json'))
if len(markers)==1: SOURCE=markers[0].parent
else:
 archives=list(INPUT.rglob('active_tb_s_synthesis_full_v1.zip')); assert len(archives)==1,archives
 attached=WORK/'attached_full_synthesis_source'
 with zipfile.ZipFile(archives[0]) as zf: zf.extractall(attached)
 SOURCE=attached/'active_tb_s_synthesis_full_v1'
PROJECT=WORK/'active_tb_s_synthesis_full_project'
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(SOURCE/'project',PROJECT)
OUT=WORK/'active_tb_s_frozen_full_v1'
MASTER_SNAPSHOT=WORK/'active_tb_s_frozen_full_checkpoint.zip'
resume_archives=list(INPUT.rglob('active_tb_s_frozen_full_checkpoint.zip'))
if len(resume_archives)>1: raise RuntimeError(resume_archives)
if OUT.exists(): shutil.rmtree(OUT)
if resume_archives:
 with zipfile.ZipFile(resume_archives[0]) as zf: zf.extractall(OUT)
else: OUT.mkdir()
print('Gold-free full synthesis source:',SOURCE,'resume:',resume_archives)


In [ ]:
boundary=json.loads((SOURCE/'SYNTHESIS_BOUNDARY.json').read_text())
assert boundary['golden_artifacts_in_dataset'] is False
assert not list(SOURCE.rglob('*mutant*'))
spec_rows=[json.loads(x) for x in (SOURCE/'spec_tasks_full.jsonl').read_text().splitlines() if x.strip()]
assert len(spec_rows)==156 and all(set(r)=={'task_id','description','header'} for r in spec_rows)
print('FULL_GOLD_FREE_SYNTHESIS_AUDIT_PASS:',len(spec_rows),'tasks')


In [ ]:
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog','yosys','z3'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','vcdvcd'],check=True)
subprocess.run([sys.executable,str(PROJECT/'typed_temporal_smoke_test.py')],check=True,cwd=PROJECT)
SEEDS=[17,29,43,59,73]
for seed in SEEDS:
 seed_out=OUT/f'seed_{seed:03d}'; seed_out.mkdir(parents=True,exist_ok=True)
 seed_snapshot=WORK/f'active_tb_s_frozen_seed_{seed:03d}_checkpoint.zip'
 partial=list(INPUT.rglob(seed_snapshot.name))
 if len(partial)>1: raise RuntimeError(partial)
 if partial and not (seed_out/'synthesis_summary.csv').exists():
  with zipfile.ZipFile(partial[0]) as zf: zf.extractall(seed_out)
 cmd=[sys.executable,str(PROJECT/'run_spec_only_synthesis.py'),
  '--tasks',str(SOURCE/'spec_tasks_full.jsonl'),
  '--candidates',*[str(SOURCE/name) for name in boundary['candidate_sources']],
  '--output',str(seed_out),'--cycle-budget','32','--scenario-pool-size','96',
  '--min-oracles','3','--min-consensus','0.75','--pseudo-mutants','24',
  '--max-formal-witness-calls','3','--formal-timeout','10','--random-seed',str(seed),
  '--resume','--snapshot-path',str(seed_snapshot),'--snapshot-every','20']
 subprocess.run(cmd,check=True,cwd=PROJECT)
 shutil.make_archive(str(MASTER_SNAPSHOT.with_suffix('')),'zip',OUT)
 print('Completed seed',seed,'master checkpoint bytes',MASTER_SNAPSHOT.stat().st_size)


In [ ]:
import pandas as pd
parts=[]
for seed in SEEDS:
 part=pd.read_csv(OUT/f'seed_{seed:03d}'/'synthesis_summary.csv'); part['seed']=seed
 assert len(part)==156*len(boundary['methods']),(seed,len(part))
 manifest=json.loads((OUT/f'seed_{seed:03d}'/'FROZEN_BEFORE_EVALUATION.json').read_text())
 assert manifest['generation_input_fields']==['task_id','description','header']
 assert manifest['random_seed']==seed
 parts.append(part)
summary=pd.concat(parts,ignore_index=True)
assert len(summary)==156*len(boundary['methods'])*len(SEEDS)
summary.to_csv(OUT/'synthesis_summary_all_seeds.csv',index=False)
report=summary.groupby(['seed','method'],as_index=False).agg(tasks=('task_id','count'),supported=('supported','sum'),mean_scenarios=('suite_scenarios','mean'))
print(report.to_string(index=False))
archive=WORK/'active_tb_s_frozen_full_v1.zip'
shutil.make_archive(str(archive.with_suffix('')),'zip',OUT)
print('Frozen full checkpoint:',archive,archive.stat().st_size,'bytes')
